# Day 11 · 딥러닝 실습 — 신경망에서 VLM까지

Colab 무료 GPU(T4)에서 PyTorch로 진행한다. 셀을 **하나씩** 실행하며 슬라이드와 짚어 나간다.

> **시작 전** — 런타임 → 런타임 유형 변경 → **T4 GPU** → 저장.

| Lab | 내용 | 슬라이드 배지 |
|---|---|---|
| 0 | 환경·GPU 확인 | — |
| 1 | NumPy·텐서 → MNIST MLP | ▶ 노트북 Lab 1 |
| 2 | CIFAR-10 MLP 한계 → CNN | ▶ 노트북 Lab 2 |
| 3 | ResNet 전이학습 + 파인튜닝 | ▶ 노트북 Lab 3 |
| 4 | CPU vs GPU 속도 | ▶ 노트북 Lab 4 |
| 5 | 전통 CV — YOLO 탐지 | ▶ 노트북 Lab 5 |
| 6 | VLM — 이미지를 말로 (NVIDIA API) | ▶ 노트북 Lab 6 |

슬라이드의 **▶ 노트북 Lab N** 배지가 이 노트북의 각 Lab을 가리킨다.

## Lab 0 · 환경·GPU 확인

In [ ]:
!nvidia-smi -L

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "· device =", device)

## Lab 1 · NumPy·텐서 → MNIST MLP

먼저 텐서가 NumPy 배열과 무엇이 같고 다른지 보고, 그 텐서로 첫 신경망을 학습시킨다.

### NumPy 배열과 텐서 — 거의 같다

In [ ]:
import numpy as np, torch

a = np.array([[1., 2.], [3., 4.]])       # NumPy 배열 (CPU)
t = torch.tensor([[1., 2.], [3., 4.]])   # 텐서
print(a)
print(t)

In [ ]:
# 모양과 연산이 같다
print("shape ", a.shape, "==", tuple(t.shape))
print("행렬 곱 np   \n", a @ a)
print("행렬 곱 torch\n", (t @ t).numpy())

In [ ]:
# 서로 오갈 수 있다
print("배열 → 텐서", torch.from_numpy(a).dtype)
print("텐서 → 배열", t.numpy().dtype)

In [ ]:
# 텐서에만 있는 것 ① GPU로 옮기기
print(t.to(device).device)

# ② 어떻게 계산됐는지 기억해 기울기를 구한다
g = torch.tensor([2.0], requires_grad=True)
(g ** 2).backward()
print("d(g^2)/dg =", g.grad.item(), "(= 2g = 4)")

### MNIST 데이터 받기

In [ ]:
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
tf = transforms.ToTensor()
train = datasets.MNIST("data", train=True,  download=True, transform=tf)
test  = datasets.MNIST("data", train=False, download=True, transform=tf)
print("학습", len(train), "· 테스트", len(test), "· 이미지 shape", train[0][0].shape)

In [ ]:
train_dl = DataLoader(train, batch_size=128, shuffle=True)
test_dl  = DataLoader(test,  batch_size=256)

### 모델 — MLP 정의

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                    # 28x28 -> 784 (한 줄로 편다)
            nn.Linear(784, 128), nn.ReLU(),  # 은닉층 + 활성화
            nn.Linear(128, 10),              # 10개 클래스 점수
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
model

### 학습 함수 — 한 에폭 도는 틀

In [ ]:
def run_epoch(model, dl, opt=None):
    train = opt is not None
    model.train(train)
    loss_fn = nn.CrossEntropyLoss()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            out = model(x)
            loss = loss_fn(out, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()   # 학습의 세 줄
        loss_sum += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item(); total += len(y)
    return loss_sum / total, correct / total

### 학습 실행

In [ ]:
for epoch in range(3):
    tr_loss, tr_acc = run_epoch(model, train_dl, opt)
    te_loss, te_acc = run_epoch(model, test_dl)
    print(f"epoch {epoch+1}: train acc {tr_acc:.3f} · test acc {te_acc:.3f}")

> **관찰** — 3에폭이면 테스트 정확도 약 0.97. `zero_grad → backward → step` 세 줄이 학습의 전부다.

## Lab 2 · CIFAR-10 — MLP의 한계, 그리고 CNN

컬러 사진(32×32×3)은 펴서 넣으면 이웃 관계가 사라진다. MLP로 해보고 한계를 본 뒤 CNN을 얹는다.

### CIFAR-10 데이터

In [ ]:
ctrain = datasets.CIFAR10("data", train=True,  download=True, transform=transforms.ToTensor())
ctest  = datasets.CIFAR10("data", train=False, download=True, transform=transforms.ToTensor())
ctrain_dl = DataLoader(ctrain, batch_size=128, shuffle=True)
ctest_dl  = DataLoader(ctest,  batch_size=256)
print("클래스", ctrain.classes)
print("이미지 shape", ctrain[0][0].shape)

### ① MLP를 사진에 — 펴서 넣는 방식

In [ ]:
class MLP_C(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                    # 3x32x32 -> 3072
            nn.Linear(3072, 256), nn.ReLU(),
            nn.Linear(256, 10))
    def forward(self, x): return self.net(x)

In [ ]:
mlp_c = MLP_C().to(device)
opt = torch.optim.Adam(mlp_c.parameters(), lr=1e-3)
for epoch in range(3):
    _, tr = run_epoch(mlp_c, ctrain_dl, opt)
    _, te = run_epoch(mlp_c, ctest_dl)
print(f"[MLP] CIFAR-10 test acc {te:.3f}   ← 사진에서는 잘 안 오른다")

### ② CNN — 합성곱으로 무늬를 보고, 풀링으로 줄인다

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32x16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 64x8x8
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(64*8*8, 128), nn.ReLU(), nn.Linear(128, 10))
    def forward(self, x): return self.head(self.feat(x))

In [ ]:
cnn = CNN().to(device)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
for epoch in range(5):
    _, tr = run_epoch(cnn, ctrain_dl, opt)
    _, te = run_epoch(cnn, ctest_dl)
    print(f"epoch {epoch+1}: test acc {te:.3f}")
print(f"[CNN] CIFAR-10 test acc {te:.3f}   ← 같은 데이터, 합성곱만 얹었다")

> **관찰** — MLP는 ~0.47에 머물고, CNN은 ~0.68. 합성곱이 이웃 픽셀의 무늬를 보기 때문.

## Lab 3 · 전이학습 — ResNet의 FC만 갈아 끼운다

ImageNet에서 배운 ResNet18을 가져와 앞쪽은 얼리고 마지막 층(fc)만 우리 문제로 바꾼다.

### 사전학습 모델 입력 규격에 맞추기

In [ ]:
tf224 = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),  # ImageNet 통계
])

In [ ]:
from torch.utils.data import Subset
tr_sub = Subset(datasets.CIFAR10("data", train=True,  transform=tf224), range(4000))
te_sub = Subset(datasets.CIFAR10("data", train=False, transform=tf224), range(1000))
tr_dl = DataLoader(tr_sub, batch_size=64, shuffle=True)
te_dl = DataLoader(te_sub, batch_size=128)
print("전이학습용 subset:", len(tr_sub), "/", len(te_sub))

### 사전학습 모델 받아 앞쪽 얼리기

In [ ]:
from torchvision import models
from torchvision.models import ResNet18_Weights

net = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for p in net.parameters():         # 앞쪽 전부 얼림
    p.requires_grad = False

In [ ]:
net.fc = nn.Linear(net.fc.in_features, 10)   # 마지막 층만 새로 (여기만 학습)
net = net.to(device)

trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
total = sum(p.numel() for p in net.parameters())
print(f"학습 파라미터 {trainable:,} / 전체 {total:,}  ({trainable/total*100:.2f}%)")

In [ ]:
opt = torch.optim.Adam(net.fc.parameters(), lr=1e-3)   # fc만 최적화
for epoch in range(2):
    _, te = run_epoch(net, tr_dl, opt)
    _, te = run_epoch(net, te_dl)
    print(f"epoch {epoch+1}: test acc {te:.3f}")

> **관찰** — 학습하는 건 전체의 1%도 안 되는 fc뿐인데 금세 앞선다. 앞쪽이 ImageNet 특징을 빌려 쓰기 때문.

### 한 걸음 더 · 파인튜닝

In [ ]:
for p in net.layer4.parameters():   # 마지막 block만 녹인다
    p.requires_grad = True
params = [p for p in net.parameters() if p.requires_grad]
print("파인튜닝 학습 파라미터:", sum(p.numel() for p in params), "개")

In [ ]:
opt = torch.optim.Adam(params, lr=1e-4)   # 특징을 망치지 않게 lr을 낮춘다
for epoch in range(2):
    _, te = run_epoch(net, tr_dl, opt)
    _, te = run_epoch(net, te_dl)
    print(f"[fine-tune] epoch {epoch+1}: test acc {te:.3f}")

## Lab 4 · CPU vs GPU — 같은 학습, 다른 시간

같은 CNN 학습 30스텝을 CPU와 GPU에서 각각 재 본다.

In [ ]:
import time

def bench(dev, steps=30):
    m = CNN().to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    xb, yb = next(iter(ctrain_dl)); xb, yb = xb.to(dev), yb.to(dev)
    if dev == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(steps):
        opt.zero_grad(); loss_fn(m(xb), yb).backward(); opt.step()
    if dev == "cuda": torch.cuda.synchronize()
    return time.time() - t0

In [ ]:
cpu_t = bench("cpu")
print(f"CPU: {cpu_t:.2f}s / 30 스텝")
if torch.cuda.is_available():
    gpu_t = bench("cuda")
    print(f"GPU: {gpu_t:.2f}s / 30 스텝  →  약 {cpu_t/gpu_t:.1f}배 빠름")
else:
    print("GPU 런타임이 아니다 — 런타임 유형을 T4로 바꾸면 비교가 나온다")

> **관찰** — GPU가 수 배~수십 배 빠르다. 단, 모델이 가벼우면 GPU가 늘 답은 아니다.

## Lab 5 · 전통 CV — YOLO로 물체 탐지

분류는 사진 전체에 라벨 하나였다. **탐지(detection)는 물체마다 박스 + 클래스**를 찾는다. YOLO는 COCO 80종을 미리 배운 모델이라 몇 줄로 바로 쓴다.

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")   # 없으면 자동으로 받는다 (~5MB) · COCO 80종

### 사진 한 장에 탐지 돌리기

In [ ]:
results = model("https://ultralytics.com/images/bus.jpg")   # 버스·사람 예시
r = results[0]
print("탐지된 물체 수:", len(r.boxes))

In [ ]:
# 무엇을 어디서 얼마나 확신하고 찾았나
for b in r.boxes:
    cls = model.names[int(b.cls)]
    conf = float(b.conf)
    xyxy = [round(v) for v in b.xyxy[0].tolist()]   # 박스 네 숫자
    print(f"{cls:10s} conf {conf:.2f}  box {xyxy}")

### 결과 그림으로 보기

In [ ]:
from PIL import Image
im = r.plot()                      # 박스가 그려진 이미지(BGR 배열)
Image.fromarray(im[:, :, ::-1])    # RGB로 뒤집어 표시

### 신뢰도·NMS 조절

In [ ]:
# conf: 이 확신 미만은 버린다 · iou: 겹치는 박스를 얼마나 합칠지(NMS)
r2 = model("https://ultralytics.com/images/bus.jpg", conf=0.5, iou=0.7)[0]
print("conf 0.25(기본):", len(r.boxes), "개")
print("conf 0.50:", len(r2.boxes), "개  ← 확신 낮은 것들이 빠진다")

> **관찰** — 탐지는 '어디에 무엇이'를 박스로 준다. NMS가 같은 물체의 겹친 박스를 하나로 합친다. 라벨(80종)이 정해져 있는 게 전통 CV의 특징 — 다음 Lab의 VLM과 대비된다.

## Lab 6 · VLM — 이미지를 말로 묻는다 (NVIDIA API)

전통 CV는 라벨을 미리 정해야 했다. VLM은 이미지에 자연어로 묻는다. NVIDIA API의 비전 모델도 OpenAI 호환 `/v1` 규격이다.

In [ ]:
%pip install -q openai

In [ ]:
import getpass, os
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...): ")

### CIFAR 테스트 이미지 한 장을 저장

In [ ]:
raw = datasets.CIFAR10("data", train=False, download=True)
pil_img, label = raw[7]
pil_big = pil_img.resize((224, 224))
pil_big.save("sample.jpg")
print("정답 라벨:", raw.classes[label])
pil_big

### 이미지 + 질문을 보낸다

In [ ]:
import base64
from openai import OpenAI
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)

img_b64 = base64.b64encode(open("sample.jpg", "rb").read()).decode()

In [ ]:
r = client.chat.completions.create(
    model="meta/llama-3.2-11b-vision-instruct",
    messages=[{"role": "user", "content": [
        {"type": "text", "text": "이 사진에 무엇이 있나? 한 문장 한국어로. 그리고 다음 중 하나만: "
                                  "airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck."},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}]}],
    max_tokens=120,
)
print(r.choices[0].message.content)

> **관찰** — VLM은 학습시키지 않았는데도 사진을 설명·분류한다. 라벨을 미리 정하지 않아도 되는 게 핵심 차이. 정밀·속도가 필요하면 전용 CV(YOLO 등), 유연함이 필요하면 VLM.

## 체크리스트

- [ ] Lab 1 — NumPy↔텐서를 확인하고, MLP로 MNIST를 학습시켰다 (test acc ~0.97)
- [ ] Lab 2 — CIFAR-10에서 MLP 한계를 보고 CNN으로 정확도를 올렸다
- [ ] Lab 3 — ResNet의 fc만 바꿔 전이학습하고, layer4를 녹여 파인튜닝했다
- [ ] Lab 4 — 같은 학습을 CPU/GPU로 돌려 속도 차를 봤다
- [ ] Lab 5 — YOLO로 물체를 탐지하고 conf·NMS를 조절했다
- [ ] Lab 6 — NVIDIA API 비전 모델에 이미지+질문을 보내 답을 받았다